# Week 5: structured data for text research

A text corpus is a collection of documents. We will organize its records in a table,
select records for a question, summarize groups, and add a simple feature from the text.

By the end, you should be able to explain:

- what one row represents and which column identifies it;
- a column's meaning, type, and missing values;
- how selection, filtering, sorting, and grouping change the table;
- how a feature connects a text to an analytical result.

We begin with short synthetic documents, then apply the same operations to the
saved OpenAlex corpus. Read the original texts when checking a result.

Start with the [conceptual slides](https://docs.google.com/presentation/d/1emSWurBlApcQHtsWBE4oWPg8kljjX4c9Azo6I5KfKDY/edit).
The final four slides contain reference patterns for use during notebook work.

## Setup — supplied
Run this cell once. It finds the course data and saves results in
`generated/week05/`. A standalone Colab copy downloads the three data files
on its first run. The setup code is supplied; it is not an exercise.

We fill the analytical cells in order during class. Later supplied cells use
those results, so the unfilled starter is not yet a complete, runnable analysis.

In [ ]:
# Supplied setup: run once. We will write the analysis below together.
from pathlib import Path
from urllib.request import urlopen
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILES = [
    'week05_text_documents.csv',
    'week05_text_documents.json',
    'openalex_berkeley_abstracts_2024_sample.csv',
]
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [p for root in roots for p in (root, root / 'compss-211a')]
ROOT = next((p for p in candidates
             if all((p / 'data' / name).exists() for name in DATA_FILES)), None)
OUTPUT = (ROOT or Path.cwd()) / 'generated/week05'
OUTPUT.mkdir(parents=True, exist_ok=True)

# A course checkout uses local files. A standalone Colab copy downloads them once.
DATA_DIR = ROOT / 'data' if ROOT is not None else OUTPUT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = 'https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/'
for name in DATA_FILES:
    destination = DATA_DIR / name
    if not destination.exists():
        with urlopen(BASE_URL + name, timeout=20) as response:
            destination.write_bytes(response.read())

pd.set_option('display.max_colwidth', 85)
print('Data:', DATA_DIR)
print('Saved results:', OUTPUT)

## 1. A file, a table, and a schema

A **schema** describes the fields and their intended meaning and types. For this
synthetic corpus:

| Field | Meaning | Intended values |
| --- | --- | --- |
| document_id | Identifies one document | Unique, present, three-character string |
| domain | Teaching category | social or physical; may repeat across documents |
| year | Publication year | Integer |
| citations | Reported citation count | Number, including zero, or missing |
| text | Original document text | String, or unavailable |

CSV stores rows with delimiters. JSON stores these example records with named
fields. Both can represent this collection. The table in Python is the object
we work on after reading a file.

**Predict:** If the CSV reader sees `001`, how might it guess the type?

In [ ]:
print((DATA_DIR / 'week05_text_documents.csv').read_text())

In [ ]:
# Together: read with inferred types and inspect the document IDs.


The ID is a label. Read it as text so the leading zeros remain part of it.
`document_id` is our **primary key**: one present, unique value per record.
The DataFrame's displayed row index is separate from this identifier.

In [ ]:
# Together: reread document_id as a string, using the schema.


In [ ]:
# Together: inspect types, missing IDs, and repeated IDs.


**Check:** A repeated domain is expected. What would a repeated document ID
make you investigate?

Record 002 has an observed citation count of zero. Record 003 has an unavailable
citation count. Records 003 and 005 have no readable text for a text analysis.
We keep those source records and decide which question they can help answer.

In [ ]:
# Together: use the supplied rule to identify present, nonblank text.


## 2. Select columns, filter rows, and sort

Before each cell, predict what a row of the result will represent and which
IDs it will contain.

**Select columns:** show the ID and original text for every document.

In [ ]:
# Your turn: select the two columns needed to inspect document texts.


**Filter rows:** keep social records from 2024. Both conditions must hold.

In [ ]:
# Together: combine the domain and year conditions, then show three columns.


**Sort:** put those records in descending citation order, then show the
first two. State the ordering rule before taking a small result.

In [ ]:
# Together: sort by citations, then take the first two records.


## 3. Categories and group summaries

Which different domains occur in the corpus? Select the category column before
asking for its distinct values.

In [ ]:
# Together: list each observed domain once.


Now make one row per domain. For the citation column:

- `size` counts document rows.
- `count` counts observed citation values, including zeros.

The recipe below gives both counts. Predict the social row before running it.

In [ ]:
# Together: use the supplied grouping recipe and explain each count.


**Filter the summary:** keep domains with at least three documents.
How is this different from keeping individual documents with three citations?

In [ ]:
# Your turn: filter the grouped table using its documents column.


## 4. Your turn: two short questions

**A.** Show the IDs and years of the physical records, oldest first.
Predict the two IDs and their order before writing code.

In [ ]:
# Your turn: select physical records, sort by year, and choose the requested columns.


**B.** Make one row per year, count its documents, and keep years with
at least three documents. Adapt the domain summary recipe.

Which table should the final condition operate on? What does a row represent then?

In [ ]:
# Your turn: adapt the grouping recipe for year and filter the resulting summary.


**Explain:** Selecting a column, filtering records, and grouping records
produce different results. Describe the output of one of your cells in one sentence.

_Write your explanation here._

## 5. Apply the operations to a real text corpus

The existing OpenAlex CSV contains 80 works from a saved 2024 teaching sample.
The source notes specify abstracts and at least one UC Berkeley-affiliated
authorship. A row represents a work. Domain is source metadata. The sample
includes different kinds of works, not only journal articles.

The supplied adapter keeps the source ID, title, original abstract, domain, and year.
These are the same roles we used in the small table.

In [ ]:
source = pd.read_csv(DATA_DIR / 'openalex_berkeley_abstracts_2024_sample.csv',
                     dtype={'openalex_id': 'string'})
corpus = source.rename(columns={'openalex_id': 'document_id',
                                'abstract': 'text', 'primary_domain': 'domain'})[
    ['document_id', 'title', 'text', 'domain', 'publication_year']
].copy()
corpus[['document_id', 'title', 'domain']].head()

In [ ]:
# Together: reuse the text/domain checks and select records for a domain comparison.


**Our question:** How often does the fragment `data` appear in abstracts in
each domain? Search for the four consecutive letters, ignoring capitalization.
`Data`, `dataset`, and `database` all match. Each abstract gets one True/False value.

This adds a **derived column** while retaining the original text. It measures
wording in the abstract. Reading a match helps us check its meaning.

In [ ]:
# Together: add the supplied literal text-match rule as a new column.


Group the selected records by domain. Count documents and True values,
then divide matches by documents in each domain.

In [ ]:
# Together: reuse grouping, count matching abstracts, and compute within-domain shares.


**Quick transfer:** show only summary rows with at least 15 documents,
ordered by share from largest to smallest. Do we filter `comparison` or `summary`?

In [ ]:
# Your turn: filter summary rows by document count, then order them by share.


## 6. A result we can trace back to text

The figure recipe is supplied. Its labels show matching abstracts / abstracts
in the domain. Proportions range from zero to one.

In [ ]:
plot_data = summary.set_index('domain')
ax = plot_data['share'].plot.barh(color='#285E78', figsize=(8, 4.5))
ax.set_xlim(0, 1)
ax.set_xlabel("Share of abstracts containing the fragment 'data'")
ax.set_ylabel('OpenAlex domain')
ax.set_title('Wording in a saved sample of 2024 works')
labels = plot_data['matches'].astype(str) + '/' + plot_data['documents'].astype(str)
ax.bar_label(ax.containers[0], labels=labels, padding=5)
plt.tight_layout()
plt.savefig(OUTPUT / 'week05_data_by_domain.png', dpi=160)
plt.show()
summary.to_csv(OUTPUT / 'week05_data_summary.csv', index=False)

Read the two abstracts below. The first matches `dataset`. The second
contains no `data` fragment. Does that tell us whether the second study used data?

In [ ]:
example = comparison[comparison['document_id'] == 'W4393372233']
print(example['title'].iloc[0])
print(example['text'].iloc[0])
example = comparison[comparison['document_id'] == 'W4404139690']
print()
print(example['title'].iloc[0])
print(example['text'].iloc[0])

**Exit:** Write one numerical finding with its denominator and one limitation.
Then explain how the document ID helps someone check it.

_Write your response here._

Next week we will build this kind of table from selected fields in a response.
Later text methods will create richer features while preserving the link to the
original documents.

## Optional: combine related tables

This extension is outside the required 120-minute path. A lookup table can store
one description for each domain. A **join** uses a shared key to attach those
details to document records. Several documents may match one lookup row.

Predict the number of document rows before running the supplied recipe. A
repeated lookup key could multiply rows, so this recipe checks that each domain
appears only once in the lookup.

In [ ]:
domains = pd.DataFrame({'domain': ['social', 'physical'],
                        'description': ['People and society', 'Physical systems']})
joined = toy.merge(domains, on='domain', how='left', validate='many_to_one')
print('Before:', len(toy), '| After:', len(joined))
joined[['document_id', 'domain', 'description']]